# Config

In [2]:
import numpy as np
CALC_FLAG = 'cpu'
GPU_FLAG = 'cuda'
CPU_FLAG = 'cpu'
PRECISION_TYPE = np.float64
# 'cpu' or 'cuda'

In [3]:
!uv pip install -q --system numba-cuda==0.4.0
from numba import config
config.CUDA_ENABLE_PYNVJITLINK = 1

In [4]:
from numba import jit, prange
#@jit(nopython=True, parallel=True)
def grad_clip_cpu(arr1d, grad_max, grad_min):
  for i in range(arr1d.shape[0]):
    if arr1d[i] > grad_max:
      arr1d[i] = grad_max
    elif arr1d[i] < -grad_max:
      arr1d[i] = -grad_max
    elif arr1d[i] < grad_min and arr1d[i] > -grad_min:
      arr1d[i] = grad_min if arr1d[i] > 0 else grad_min
  return arr1d

def grad_clip(arr, grad_max=1e-1, grad_min=1e-7):
  arr1d = arr.reshape(-1)

  if CALC_FLAG == CPU_FLAG:
    return grad_clip_cpu(arr1d, grad_max, grad_min)
  elif CALC_FLAG == GPU_FLAG:
    raise NotImplementedError

  arr = arr1d.reshape(arr.shape)

# YOLOv4 backbone: CSPDarknet53

## Conv2D layer

### Conv2D core

#### Functions

In [5]:
from numba import jit, prange
#@jit(nopython=True, parallel=True)
def conv2d_cpu(input4d, kernel4d, bias1d, output4d, stride, padding, dilation):
  for im in range(output4d.shape[0]):
    for c_out in range(output4d.shape[1]):
      for x in range(output4d.shape[2]):
        for y in range(output4d.shape[3]):
          sum = 0
          for x_k in range(kernel4d.shape[2]):
            for y_k in range(kernel4d.shape[3]):
              x_in = x * stride + x_k * dilation - padding
              y_in = y * stride + y_k * dilation - padding
              if (0 <= x_in < input4d.shape[2]) and (0 <= y_in < input4d.shape[3]):
                for c_in in range(input4d.shape[1]):
                  sum += input4d[im, c_in, x_in, y_in] * kernel4d[c_out, c_in, x_k, y_k]
          output4d[im, c_out, x, y] = sum + bias1d[c_out]

#@jit(nopython=True, parallel=True)
def conv2d_biasBack_cpu(d_output4d, bias1d):
  for c_out in range(bias1d.shape[0]):
    sum = 0
    for im in range(d_output4d.shape[0]):
      for x in range(d_output4d.shape[2]):
        for y in range(d_output4d.shape[3]):
          sum += d_output4d[im, c_out, x, y]
    bias1d[c_out] = sum

#@jit(nopython=True, parallel=True)
def flipKernel_cpu(kernel4d, output4d):
  for c_out in range(kernel4d.shape[0]):
    for c_in in range(kernel4d.shape[1]):
      for x_k in range(kernel4d.shape[2]):
        for y_k in range(kernel4d.shape[3]):
          output4d[c_out, c_in, x_k, y_k] = kernel4d[c_out, c_in, kernel4d.shape[2]-x_k-1, kernel4d.shape[3]-y_k-1]

#### Classes

In [6]:
import numpy as np
class NNLayer4D:
  def __init__(self, in_channels):
    self.in_channels = in_channels
    self.input = None
    self.output = None

  def forward(self, input):
    if len(input.shape) != 4:
      raise ValueError(f"Input has to be a 4D array, got {len(input.shape)} instead.")
    elif input.shape[1] != self.in_channels:
      raise ValueError(f"Input has to have {self.in_channels} channels, got {input.shape[1]} instead.")
    # to save the input for backpropagation
    self.input = input

  def backward(self, d_output):
    pass

  @classmethod
  def class_name(cls):
    pass

  def save_pickle(self):
    pass

  @classmethod
  def load_pickle(self, data):
    pass

class NNConv2D(NNLayer4D):
  def __init__(self, in_channels, out_channels, kernel_size, stride, padding, dilation):
    super().__init__(in_channels)
    self.out_channels = out_channels
    self.kernel_size = kernel_size
    self.stride = stride
    self.padding = padding
    self.dilation = dilation

    n_para = out_channels * in_channels * kernel_size**2
    self.kernel = np.random.normal(0, np.sqrt(2/n_para), \
     (out_channels, in_channels, kernel_size, kernel_size)).astype(PRECISION_TYPE) # Kaiming initialization
    self.bias = np.random.normal(0, np.sqrt(2/out_channels), out_channels).astype(PRECISION_TYPE)
    self.d_kernel = np.zeros_like(self.kernel)
    self.d_bias = np.zeros_like(self.bias)

  def forward(self, input):
    super().forward(input)

    output_shape = (input.shape[0], self.out_channels,
              (input.shape[2] + 2 * self.padding - self.dilation * (self.kernel_size - 1) - 1) // self.stride + 1,
              (input.shape[3] + 2 * self.padding - self.dilation * (self.kernel_size - 1) - 1) // self.stride + 1)
    self.output = np.zeros(output_shape, dtype=PRECISION_TYPE)

    if CALC_FLAG == CPU_FLAG:
      conv2d_cpu(self.input, self.kernel, self.bias, self.output, self.stride, self.padding, self.dilation)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    return self.output

  def backward(self, d_output):
    # d(bias) = sum of each 3D image.
    if CALC_FLAG == CPU_FLAG:
      conv2d_biasBack_cpu(d_output, self.d_bias)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # d(kernel) = conv(inputT, d_outT)T
    # note: input [N, c_in, x, y], d_out [N, c_out, x, y], kernel [c_out, c_in, x, y]
    # idea: if we swap the 1st and 2nd axis, we get that d_out [c_out, N] can be used as a kernel for input [c_in, N]
    # to calculate swapped version of kernel [c_in, c_out].
    bias_zero = np.zeros(self.out_channels, dtype=PRECISION_TYPE)
    input_T = self.input.transpose(1, 0, 2, 3)
    d_output_T = d_output.transpose(1, 0, 2, 3)
    d_kernel_T = self.d_kernel.transpose(1, 0, 2, 3)

    if CALC_FLAG == CPU_FLAG:
      conv2d_cpu(input_T, d_output_T, bias_zero, d_kernel_T, self.stride, self.padding, self.dilation)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # d(input) = conv(d_out, flipped180(kernelT))
    bias_zero = np.zeros(self.in_channels, dtype=PRECISION_TYPE)
    d_input = np.zeros(self.input.shape, dtype=PRECISION_TYPE)
    flipped_kernel = np.zeros_like(self.kernel)

    flipKernel_cpu(self.kernel, flipped_kernel)
    flipped_kernel = flipped_kernel.transpose(1, 0, 2, 3)

    if CALC_FLAG == CPU_FLAG:
      conv2d_cpu(d_output, flipped_kernel, bias_zero, d_input, self.stride, self.padding, self.dilation)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # gradient clip
    grad_clip(self.d_kernel)
    grad_clip(self.d_bias)
    grad_clip(d_input)

    return d_input

  @classmethod
  def class_name(cls):
    return "NNConv2D"

  def save_pickle(self):
    return {
        'class': self.class_name(),
        'in_channels': self.in_channels,
        'out_channels': self.out_channels,
        'kernel_size': self.kernel_size,
        'stride': self.stride,
        'padding': self.padding,
        'dilation': self.dilation,

        'kernel': self.kernel,
        'bias': self.bias,
        'd_kernel': self.d_kernel,
        'd_bias': self.d_bias,

        'input': self.input,
        'output': self.output
    }

  @classmethod
  def load_pickle(cls, data):
    obj = cls(data['in_channels'], data['out_channels'], data['kernel_size'], data['stride'], data['padding'], data['dilation'])
    obj.kernel = data['kernel']
    obj.bias = data['bias']
    obj.d_kernel = data['d_kernel']
    obj.d_bias = data['d_bias']
    obj.input = data['input']
    obj.output = data['output']
    return obj

#### Test

In [7]:
# import pickle
# input = np.random.rand(1, 3, 32, 32)
# d_out = np.random.rand(1, 512, 32, 32)

# conv = NNConv2D(3, 512, 3, 1, 1, 1)
# output = conv.forward(input)
# d_input = conv.backward(d_out)

# print(np.mean(d_input), np.min(d_input), np.max(d_input))

### Batch normalization

#### Functions

In [8]:
from numba import jit
#@jit(nopython=True, parallel=True)
def meanvar_cpu(image4d, mean1d, var1d):
  for c in range(image4d.shape[1]):
    # mean
    mean = 0
    for k in range(image4d.shape[0]):
      for x in range(image4d.shape[2]):
        for y in range(image4d.shape[3]):
          mean += image4d[k, c, x, y]
    mean /= (image4d.shape[2] * image4d.shape[3] * image4d.shape[0])
    mean1d[c] = mean

    # var
    var = 0
    for k in range(image4d.shape[0]):
      for x in range(image4d.shape[2]):
        for y in range(image4d.shape[3]):
          var += (image4d[k, c, x, y] - mean) ** 2
    var1d[c] = var / (image4d.shape[2] * image4d.shape[3] * image4d.shape[0])

#@jit(nopython=True, parallel=True)
def bn_forward_cpu(input4d, weight1d, bias1d, output4d, Mean1d, Var1d, stab):
  for k in range(input4d.shape[0]):
    for c in range(input4d.shape[1]):
      for x in range(input4d.shape[2]):
        for y in range(input4d.shape[3]):
          output4d[k,c,x,y] = weight1d[c]*(input4d[k,c,x,y] - Mean1d[c]) / np.sqrt(Var1d[c] + stab)+bias1d[c]

#@jit(nopython=True, parallel=True)
def bn_forward_intermed_cpu(input4d, output4d, Mean1d, Var1d, stab):
  for k in range(input4d.shape[0]):
    for c in range(input4d.shape[1]):
      for x in range(input4d.shape[2]):
        for y in range(input4d.shape[3]):
          output4d[k,c,x,y] = (input4d[k,c,x,y] - Mean1d[c]) / np.sqrt(Var1d[c] + stab)

#@jit(nopython=True, parallel=True)
def bn_back_wgrads_cpu(d_output4d, prev_output4d, d_weight1d, d_bias1d):
  for c in range(d_output4d.shape[1]):
    sumW, sumB = 0, 0
    for k in range(d_output4d.shape[0]):
      for x in range(d_output4d.shape[2]):
        for y in range(d_output4d.shape[3]):
          sumB += d_output4d[k,c,x,y]
          sumW += prev_output4d[k,c,x,y] * d_output4d[k,c,x,y]
    d_weight1d[c] = sumW
    d_bias1d[c] = sumB

#@jit(nopython=True, parallel=True)
def bn_back_outputpass_cpu(d_output4d, weight1d, d_output4d_out):
  for k in range(d_output4d.shape[0]):
    for c in range(d_output4d.shape[1]):
      for x in range(d_output4d.shape[2]):
        for y in range(d_output4d.shape[3]):
          d_output4d_out[k,c,x,y] = weight1d[c] * d_output4d[k,c,x,y]

#@jit(nopython=True, parallel=True)
def bn_back_input_cpu(dx_hat4d, x_hat4d, Var1d, d_input4d, d_prod1d, d_sum1d, weight1d, stab):
  # calculate stuff
  N = d_input4d.shape[0] * d_input4d.shape[2] * d_input4d.shape[3]

  # final
  for k in range(d_input4d.shape[0]):
    for c in range(d_input4d.shape[1]):
      for x in range(d_input4d.shape[2]):
        for y in range(d_input4d.shape[3]):
          d_input4d[k,c,x,y] = weight1d[c]\
                      * (N * dx_hat4d[k,c,x,y] - d_sum1d[c] - x_hat4d[k,c,x,y] * d_prod1d[c])\
                      / N / np.sqrt(Var1d[c] + stab)

#### Classes

In [9]:
class BatchNorm2D(NNLayer4D):
  def __init__(self, in_channels, stab=1e-3):
    super().__init__(in_channels)
    self.stab = stab
    self.mean = np.zeros((in_channels), dtype=PRECISION_TYPE)
    self.var = np.zeros((in_channels), dtype=PRECISION_TYPE)

    self.weight = np.random.rand(in_channels).astype(PRECISION_TYPE)
    self.bias = np.random.rand(in_channels).astype(PRECISION_TYPE)
    self.d_weight = np.zeros_like(self.weight)
    self.d_bias = np.zeros_like(self.bias)

  def forward(self, input):
    super().forward(input)

    # calculate batch norm
    self.output = np.zeros_like(input)

    if CALC_FLAG == CPU_FLAG:
      meanvar_cpu(input, self.mean, self.var)
      bn_forward_cpu(input, self.weight, self.bias, self.output, self.mean, self.var, self.stab)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    return self.output

  def backward(self, d_output):
    super().forward(d_output)

    d_input = np.zeros_like(d_output)
    d_prod = np.zeros((self.in_channels))
    d_sum = np.zeros((self.in_channels))
    x_hat = np.zeros_like(d_output)
    dx_hat = np.zeros_like(d_output)

    if CALC_FLAG == CPU_FLAG:
      # recalculate intermediate output
      bn_forward_intermed_cpu(self.input, x_hat, self.mean, self.var, self.stab)

      # calculate w, b gradients
      bn_back_wgrads_cpu(d_output, x_hat, self.d_weight, self.d_bias)

      # calculate d_out * weight
      bn_back_outputpass_cpu(d_output, self.weight, dx_hat)

      # calculate d_prod, d_sum
      bn_back_wgrads_cpu(dx_hat, x_hat, d_prod, d_sum)

      # calculate input gradients
      bn_back_input_cpu(dx_hat, x_hat, self.var, d_input, d_prod, d_sum, self.weight, self.stab)

    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # gradient clip
    grad_clip(self.d_weight)
    grad_clip(self.d_bias)
    grad_clip(d_input)

    return d_input

  @classmethod
  def class_name(cls):
    return "BatchNorm2D"

  def save_pickle(self):
    return {
        'class': self.class_name(),
        'in_channels': self.in_channels,
        'stab': self.stab,
        'mean': self.mean,
        'var': self.var,
        'weight': self.weight,
        'bias': self.bias,
        'd_weight': self.d_weight,
        'd_bias': self.d_bias,
        'input': self.input,
        'output': self.output
    }

  @classmethod
  def load_pickle(cls, data):
    obj = cls(data['in_channels'], data['stab'])
    obj.mean = data['mean']
    obj.var = data['var']
    obj.weight = data['weight']
    obj.bias = data['bias']
    obj.d_weight = data['d_weight']
    obj.d_bias = data['d_bias']
    obj.input = data['input']
    obj.output = data['output']
    return obj

#### Test

In [10]:
# import pickle
# input = np.random.rand(1, 3, 32, 32)
# d_out = np.random.rand(1, 3, 32, 32)

# bn = BatchNorm2D(3)
# with open('bn.pickle', 'wb') as f:
#   pickle.dump(bn.save_pickle(), f)
# output = bn.forward(input)
# d_input = bn.backward(d_out)

# bn1 = None
# with open('bn.pickle', 'rb') as f:
#   data = pickle.load(f)
#   bn1 = BatchNorm2D.load_pickle(data)
# output1 = bn1.forward(input)
# d_input1 = bn1.backward(d_out)

# print(np.allclose(output, output1))
# print(np.allclose(d_input, d_input1))
# print(np.mean(d_input))

### LeakyReLU

#### Functions

In [11]:
#@jit(nopython=True, parallel=True)
def leakyrelu_cpu(input4d, output4d, alpha):
  for i in range(input4d.shape[0]):
    for j in range(input4d.shape[1]):
      for k in range(input4d.shape[2]):
        for l in range(input4d.shape[3]):
          if input4d[i,j,k,l] < 0:
            output4d[i,j,k,l] = alpha * input4d[i,j,k,l]
          else:
            output4d[i,j,k,l] = input4d[i,j,k,l]

#@jit(nopython=True, parallel=True)
def leakyrelu_back_cpu(d_output4d, prev_input4d, d_input4d, alpha):
  for i in range(d_output4d.shape[0]):
    for j in range(d_output4d.shape[1]):
      for k in range(d_output4d.shape[2]):
        for l in range(d_output4d.shape[3]):
          if prev_input4d[i,j,k,l] < 0:
            d_input4d[i,j,k,l] = alpha * d_output4d[i,j,k,l]
          else:
            d_input4d[i,j,k,l] = d_output4d[i,j,k,l]

#### Classes

In [12]:
class LeakyReLU(NNLayer4D):
  def __init__(self, in_channels, alpha=0.1):
    super().__init__(in_channels)
    self.alpha = alpha

  def forward(self, input):
    super().forward(input)

    self.output = np.zeros_like(input)

    if CALC_FLAG == CPU_FLAG:
      leakyrelu_cpu(input, self.output, self.alpha)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    return self.output

  def backward(self, d_output):
    d_input = np.zeros_like(self.input)

    if CALC_FLAG == CPU_FLAG:
      leakyrelu_back_cpu(d_output, self.input, d_input, self.alpha)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    grad_clip(d_input)

    return d_input

  @classmethod
  def class_name(cls):
    return "LeakyReLU"

  def save_pickle(self):
    return {
        'class': self.class_name(),
        'in_channels': self.in_channels,
        'alpha': self.alpha,
        'input': self.input,
        'output': self.output
    }

  @classmethod
  def load_pickle(cls, data):
    obj = cls(data['in_channels'], data['alpha'])
    obj.input = data['input']
    obj.output = data['output']
    return obj

#### Test

In [13]:
# import numpy as np
# import pickle

# input = np.random.rand(1, 3, 32, 32)
# d_out = np.random.rand(1, 3, 32, 32)

# relu = LeakyReLU(3, alpha=0.1)

# with open('leaky_relu.pickle', 'wb') as f:
#     pickle.dump(relu.save_pickle(), f)

# output = relu.forward(input)
# d_input = relu.backward(d_out)

# with open('leaky_relu.pickle', 'rb') as f:
#     data = pickle.load(f)
#     relu1 = LeakyReLU.load_pickle(data)

# output1 = relu1.forward(input)
# d_input1 = relu1.backward(d_out)

# print(np.allclose(output, output1))
# print(np.allclose(d_input, d_input1))

### Conv2D layer

#### Classes

In [14]:
class Conv2D(NNLayer4D):
  def __init__(self, in_channels, out_channels, kernel_size, stride, padding, dilation):
    super().__init__(in_channels)

    self.out_channels = out_channels
    self.kernel_size = kernel_size
    self.stride = stride
    self.padding = padding
    self.dilation = dilation

    self.conv = NNConv2D(in_channels, out_channels, kernel_size, stride, padding, dilation)
    self.bn = BatchNorm2D(out_channels)
    self.relu = LeakyReLU(out_channels)

  def forward(self, input):
    super().forward(input)
    output = self.conv.forward(input)
    output = self.bn.forward(output)
    output = self.relu.forward(output)
    self.output = output
    return output

  def backward(self, d_output):
    d_output = self.relu.backward(d_output)
    d_output = self.bn.backward(d_output)
    d_output = self.conv.backward(d_output)
    return d_output

  @classmethod
  def class_name(cls):
    return "Conv2D"

  def save_pickle(self):
    return {
        'class': self.class_name(),

        'in_channels': self.in_channels,
        'out_channels': self.out_channels,
        'kernel_size': self.kernel_size,
        'stride': self.stride,
        'padding': self.padding,
        'dilation': self.dilation,

        'conv': self.conv.save_pickle(),
        'bn': self.bn.save_pickle(),
        'relu': self.relu.save_pickle(),

        'input': self.input,
        'output': self.output
    }

  @classmethod
  def load_pickle(cls, data):
    obj = cls(data['in_channels'], data['out_channels'], data['kernel_size'], data['stride'], data['padding'], data['dilation'])
    obj.conv = NNConv2D.load_pickle(data['conv'])
    obj.bn = BatchNorm2D.load_pickle(data['bn'])
    obj.relu = LeakyReLU.load_pickle(data['relu'])
    obj.input = data['input']
    obj.output = data['output']
    return obj

#### Test

In [15]:
# import numpy as np
# import pickle

# input = np.random.rand(1, 3, 32, 32)

# conv = Conv2D(3, 6, kernel_size=3, stride=1, padding=0, dilation=1)

# with open('conv.pickle', 'wb') as f:
#     pickle.dump(conv.save_pickle(), f)

# output = conv.forward(input)
# d_out = np.random.rand(*output.shape)
# d_input = conv.backward(d_out)

# with open('conv.pickle', 'rb') as f:
#     data = pickle.load(f)
#     conv1 = Conv2D.load_pickle(data)

# output1 = conv1.forward(input)
# d_input1 = conv1.backward(d_out)

# print(np.allclose(output, output1))
# print(np.allclose(d_input, d_input1))
# print(np.mean(d_input))

## Residual block

### Classes

In [16]:
class CSPResidualBlock(NNLayer4D):
  def __init__(self, in_channels):
    if in_channels % 4 != 0:
      raise ValueError("in_channels must be divisible by 4")
    super().__init__(in_channels)

    split_channels = in_channels // 2
    self.conv1 = Conv2D(split_channels, split_channels // 2, 1, 1, 0, 1)
    self.conv2 = Conv2D(split_channels // 2, split_channels, 3, 1, 1, 1)
    self.conv3 = Conv2D(in_channels, in_channels, 1, 1, 0, 1)

  def forward(self, input):
    # check
    super().forward(input)

    # execute
    input1, input2 = np.split(input, 2, axis=1)

    input1 = self.conv1.forward(input1)
    input1 = self.conv2.forward(input1)

    output = np.concatenate([input1, input2], axis=1)
    output = self.conv3.forward(output)
    self.output = output

    return output

  def backward(self, d_output):
    dx3 = self.conv3.backward(d_output)
    dx1, dx2 = np.split(dx3, 2, axis=1)

    dx1 = self.conv2.backward(dx1)
    dx1 = self.conv1.backward(dx1)

    d_input = np.concatenate([dx1, dx2], axis=1)

    return d_input

  @classmethod
  def class_name(cls):
    return "CSPResidualBlock"

  def save_pickle(self):
    return{
        'class': self.class_name(),
        'in_channels': self.in_channels,
        'conv1': self.conv1.save_pickle(),
        'conv2': self.conv2.save_pickle(),
        'conv3': self.conv3.save_pickle(),

        'input': self.input,
        'output': self.output
    }

  @classmethod
  def load_pickle(cls, data):
    obj = cls(data['in_channels'])
    obj.conv1 = Conv2D.load_pickle(data['conv1'])
    obj.conv2 = Conv2D.load_pickle(data['conv2'])
    obj.conv3 = Conv2D.load_pickle(data['conv3'])
    obj.input = data['input']
    obj.output = data['output']
    return obj

### Test

In [17]:
# import numpy as np
# import pickle

# input = np.random.rand(1, 8, 32, 32)
# print(np.mean(input), np.min(input), np.max(input))

# res = CSPResidualBlock(8)
# res1 = None

# with open('resblock.pickle', 'wb') as f:
#     pickle.dump(res.save_pickle(), f)

# output = res.forward(input)
# d_out = np.random.rand(*output.shape)
# print(np.mean(d_out), np.min(d_out), np.max(d_out))
# din = res.backward(d_out)

# with open('resblock.pickle', 'rb') as f:
#     res1 = CSPResidualBlock.load_pickle(pickle.load(f))

# output1 = res1.forward(input)
# dinT = res1.backward(d_out)

# print(np.mean(np.abs(din - dinT)))
# print(np.mean(din))

## CSPDarknet53

#### Functions

In [18]:
from numba import jit, prange
#@jit(nopython=True, parallel=True)
def add_cpu(input4d_1, input4d_2, output4d):
  for k in range(input4d_1.shape[0]):
    for c in range(input4d_1.shape[1]):
      for x in range(input4d_1.shape[2]):
        for y in range(input4d_1.shape[3]):
          output4d[k,c,x,y] = input4d_1[k,c,x,y] + input4d_2[k,c,x,y]

#### Implementation

In [19]:
class NN:
  def __init__(self, layers = None):
    self.input = None
    self.output = None
    self.layers = layers
    # Note: this piece of code is so dumb and so inconsistent so...

    # if layers is None:
    #   self.layers = []
    # elif isinstance(layers, NNLayer4D):
    #   self.layers = [layers]
    # elif isinstance(layers, list) and ((len(layers) > 0 and all([isinstance(layer, NNLayer4D) for layer in layers])) or len(layers) == 0):
    #   self.layers = layers
    # else:
    #   raise TypeError("Unsupported layer type")

  def forward(self, input):
    self.input = input
    output = input
    for i in range(len(self.layers)):
      output = self.layers[i].forward(output)
    self.output = output
    return self.output

  def backward(self, d_output):
    for i in range(len(self.layers)-1, -1, -1):
      d_output = self.layers[i].backward(d_output)
      #print(np.mean(d_output), np.min(d_output), np.max(d_output))
      grad_clip(d_output, grad_max=1e1)
      #print(np.mean(d_output), np.min(d_output), np.max(d_output))
    return d_output # d_input

  @classmethod
  def class_name(cls):
    return "NeuralNetwork"

  def save_pickle(self):
    return {
        'class': self.class_name(),
        'layers': [layer.save_pickle() for layer in self.layers],
        'input': self.input,
        'output': self.output
    }

  @classmethod
  def load_pickle(cls, data):
    layers = []
    for layer in data['layers']:
      if layer['class'] == Conv2D.class_name():
        layers.append(Conv2D.load_pickle(layer))
      elif layer['class'] == CSPResidualBlock.class_name():
        layers.append(CSPResidualBlock.load_pickle(layer))
      else:
        raise NotImplementedError(f"class {layer['class']}")
    obj = cls(layers)
    obj.input = data['input']
    obj.output = data['output']
    return obj

class CSPDarknet53:
  def __init__(self, in_channels):
    self.in_channels = in_channels
    start_channels = 2 # for testing purpose!!!
    self.csp_p3 = NN([
        Conv2D(in_channels, start_channels, 3, 1, 1, 1),
        Conv2D(start_channels, start_channels * 2, 3, 2, 1, 1),
        CSPResidualBlock(start_channels * 2),
        Conv2D(start_channels * 2, start_channels * 4, 3, 2, 1, 1),
        CSPResidualBlock(start_channels * 4),
        CSPResidualBlock(start_channels * 4),
        Conv2D(start_channels * 4, start_channels * 8, 3, 2, 1, 1),
        CSPResidualBlock(start_channels * 8),
        CSPResidualBlock(start_channels * 8),
        CSPResidualBlock(start_channels * 8),
        CSPResidualBlock(start_channels * 8),
        CSPResidualBlock(start_channels * 8),
        CSPResidualBlock(start_channels * 8),
        CSPResidualBlock(start_channels * 8),
        CSPResidualBlock(start_channels * 8)
    ])

    self.csp_p4 = NN([
        Conv2D(start_channels * 8, start_channels * 16, 3, 2, 1, 1),
        CSPResidualBlock(start_channels * 16),
        CSPResidualBlock(start_channels * 16),
        CSPResidualBlock(start_channels * 16),
        CSPResidualBlock(start_channels * 16),
        CSPResidualBlock(start_channels * 16),
        CSPResidualBlock(start_channels * 16),
        CSPResidualBlock(start_channels * 16),
        CSPResidualBlock(start_channels * 16)
    ])

    self.csp_p5 = NN([
        Conv2D(start_channels * 16, start_channels * 32, 3, 2, 1, 1),
        CSPResidualBlock(start_channels * 32),
        CSPResidualBlock(start_channels * 32),
        CSPResidualBlock(start_channels * 32),
        CSPResidualBlock(start_channels * 32)
    ])


  def forward(self, input):
    output = self.csp_p3.forward(input)
    output2 = self.csp_p4.forward(output)
    output3 = self.csp_p5.forward(output2)
    return output, output2, output3

  def backward(self, do_p3, do_p4, do_p5):
    di_p5 = self.csp_p5.backward(do_p5)

    if CALC_FLAG == CPU_FLAG:
      add_cpu(do_p4, di_p5, do_p4)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    di_p4 = self.csp_p4.backward(do_p4)

    if CALC_FLAG == CPU_FLAG:
      add_cpu(do_p3, di_p4, do_p3)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    di = self.csp_p3.backward(do_p3)

    return di, di_p4, di_p5

  @classmethod
  def class_name(cls):
    return "CSPDarknet53"

  def save_pickle(self):
    return {
        'class': self.class_name(),
        'in_channels': self.in_channels,
        'csp_p3': self.csp_p3.save_pickle(),
        'csp_p4': self.csp_p4.save_pickle(),
        'csp_p5': self.csp_p5.save_pickle()
    }

  @classmethod
  def load_pickle(cls, data):
    obj = cls(data['in_channels'])
    obj.csp_p3 = NN.load_pickle(data['csp_p3'])
    obj.csp_p4 = NN.load_pickle(data['csp_p4'])
    obj.csp_p5 = NN.load_pickle(data['csp_p5'])
    return obj


#### Test

In [20]:
# LEGACY CODE BY CHATGPT

# import numpy as np

# # ------------------------ Core Layers ------------------------

# class Conv2D:
#     def __init__(self, in_channels, out_channels, kernel_size, stride, padding, dilation):
#         self.in_channels = in_channels
#         self.out_channels = out_channels
#         self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
#         self.stride = stride
#         self.padding = padding
#         self.dilation = dilation

#         kh, kw = self.kernel_size
#         self.weight = np.random.randn(out_channels, in_channels, kh, kw) * np.sqrt(2 / (in_channels * kh * kw))
#         self.bias = np.zeros(out_channels)
#         self.d_weight = np.zeros_like(self.weight)
#         self.d_bias = np.zeros_like(self.bias)

#     def forward(self, input):
#         self.input = input
#         n, c_in, h, w = input.shape
#         c_out, _, kh, kw = self.weight.shape

#         out_h = (h + 2 * self.padding - self.dilation * (kh - 1) - 1) // self.stride + 1
#         out_w = (w + 2 * self.padding - self.dilation * (kw - 1) - 1) // self.stride + 1

#         output = np.zeros((n, c_out, out_h, out_w))
#         padded_input = np.pad(input, ((0,0), (0,0), (self.padding, self.padding), (self.padding, self.padding)), mode='constant')

#         for i in range(out_h):
#             for j in range(out_w):
#                 x_start = i * self.stride
#                 y_start = j * self.stride
#                 x_end = x_start + kh * self.dilation
#                 y_end = y_start + kw * self.dilation

#                 region = padded_input[:, :, x_start:x_end:self.dilation, y_start:y_end:self.dilation]
#                 for k in range(c_out):
#                     output[:, k, i, j] = np.sum(region * self.weight[k], axis=(1,2,3))
#         output += self.bias[None, :, None, None]
#         return output

#     def backward(self, d_out):
#         input = self.input
#         n, c_in, h, w = input.shape
#         c_out, _, kh, kw = self.weight.shape
#         _, _, out_h, out_w = d_out.shape

#         padded_input = np.pad(input, ((0,0), (0,0), (self.padding, self.padding), (self.padding, self.padding)), mode='constant')
#         d_input = np.zeros_like(padded_input)
#         self.d_weight.fill(0)
#         self.d_bias = np.sum(d_out, axis=(0, 2, 3))

#         for i in range(out_h):
#             for j in range(out_w):
#                 x_start = i * self.stride
#                 y_start = j * self.stride
#                 x_end = x_start + kh * self.dilation
#                 y_end = y_start + kw * self.dilation

#                 region = padded_input[:, :, x_start:x_end:self.dilation, y_start:y_end:self.dilation]
#                 for k in range(c_out):
#                     self.d_weight[k] += np.sum(region * d_out[:, k:k+1, i:i+1, j:j+1], axis=0)
#                     d_input[:, :, x_start:x_end:self.dilation, y_start:y_end:self.dilation] += self.weight[k] * d_out[:, k:k+1, i:i+1, j:j+1]

#         if self.padding > 0:
#             d_input = d_input[:, :, self.padding:-self.padding, self.padding:-self.padding]
#         return d_input


# class BatchNorm2D:
#     def __init__(self, channels, momentum=0.1, epsilon=1e-5):
#         self.channels = channels
#         self.momentum = momentum
#         self.epsilon = epsilon
#         self.gamma = np.ones((1, channels, 1, 1))
#         self.beta = np.zeros((1, channels, 1, 1))
#         self.running_mean = np.zeros((1, channels, 1, 1))
#         self.running_var = np.ones((1, channels, 1, 1))

#     def forward(self, x):
#         self.input = x
#         self.mean = np.mean(x, axis=(0,2,3), keepdims=True)
#         self.var = np.var(x, axis=(0,2,3), keepdims=True)
#         self.x_hat = (x - self.mean) / np.sqrt(self.var + self.epsilon)
#         self.output = self.gamma * self.x_hat + self.beta
#         return self.output

#     def backward(self, d_out):
#         N, C, H, W = d_out.shape
#         x_mu = self.input - self.mean
#         std_inv = 1. / np.sqrt(self.var + self.epsilon)

#         d_xhat = d_out * self.gamma
#         d_var = np.sum(d_xhat * x_mu * -0.5 * std_inv**3, axis=(0,2,3), keepdims=True)
#         d_mean = np.sum(d_xhat * -std_inv, axis=(0,2,3), keepdims=True) + d_var * np.mean(-2. * x_mu, axis=(0,2,3), keepdims=True)
#         d_input = d_xhat * std_inv + d_var * 2 * x_mu / (N*H*W) + d_mean / (N*H*W)

#         return d_input


# class LeakyReLU:
#     def __init__(self, channels, alpha=0.1):
#         self.alpha = alpha

#     def forward(self, x):
#         self.input = x
#         return np.where(x > 0, x, self.alpha * x)

#     def backward(self, d_out):
#         dx = np.where(self.input > 0, 1, self.alpha)
#         return d_out * dx

# # ------------------------ Building Blocks ------------------------

# class ConvBNReLU:
#     def __init__(self, in_channels, out_channels, kernel_size, stride, padding, dilation=1):
#         self.conv = Conv2D(in_channels, out_channels, kernel_size, stride, padding, dilation)
#         self.bn = BatchNorm2D(out_channels)
#         self.relu = LeakyReLU(out_channels)

#     def forward(self, x):
#         x = self.conv.forward(x)
#         x = self.bn.forward(x)
#         x = self.relu.forward(x)
#         return x

#     def backward(self, d_out):
#         d_out = self.relu.backward(d_out)
#         d_out = self.bn.backward(d_out)
#         d_out = self.conv.backward(d_out)
#         return d_out


# class CSPResidualBlock:
#     def __init__(self, in_channels):
#         assert in_channels % 4 == 0
#         split_channels = in_channels // 2
#         self.conv1 = ConvBNReLU(split_channels, split_channels // 2, 1, 1, 0)
#         self.conv2 = ConvBNReLU(split_channels // 2, split_channels, 3, 1, 1)
#         self.conv3 = ConvBNReLU(in_channels, in_channels, 1, 1, 0)

#     def forward(self, x):
#         self.input = x
#         input1, input2 = np.split(x, 2, axis=1)
#         input1 = self.conv1.forward(input1)
#         input1 = self.conv2.forward(input1)
#         output = np.concatenate([input1, input2], axis=1)
#         output = self.conv3.forward(output)
#         self.output = output
#         return output

#     def backward(self, d_out):
#         d_out = self.conv3.backward(d_out)
#         dx1, dx2 = np.split(d_out, 2, axis=1)
#         dx1 = self.conv2.backward(dx1)
#         dx1 = self.conv1.backward(dx1)
#         return np.concatenate([dx1, dx2], axis=1)


# class CSPStage:
#     def __init__(self, in_channels, num_blocks):
#         self.downsample = ConvBNReLU(in_channels, in_channels * 2, 3, 2, 1)
#         self.blocks = [CSPResidualBlock(in_channels * 2) for _ in range(num_blocks)]

#     def forward(self, x):
#         x = self.downsample.forward(x)
#         for block in self.blocks:
#             x = block.forward(x)
#         return x

#     def backward(self, d_out):
#         for block in reversed(self.blocks):
#             d_out = block.backward(d_out)
#         return self.downsample.backward(d_out)


# class CSPDarknet53:
#     def __init__(self):
#         self.stem = ConvBNReLU(3, 32, 3, 1, 1)
#         self.stages = [
#             CSPStage(32, 1),    # output: 64
#             CSPStage(64, 2),    # output: 128
#             CSPStage(128, 8),   # output: 256
#             CSPStage(256, 8),   # output: 512
#             CSPStage(512, 4),   # output: 1024
#         ]

#     def forward(self, x):
#         x = self.stem.forward(x)
#         for stage in self.stages:
#             x = stage.forward(x)
#         return x

#     def backward(self, d_out):
#         for stage in reversed(self.stages):
#             d_out = stage.backward(d_out)
#         return self.stem.backward(d_out)

# # ------------------------ Test Script ------------------------

# if __name__ == "__main__":
#     np.random.seed(0)
#     input = np.random.randn(1, 3, 64, 64)
#     model = CSPDarknet53()

#     output = model.forward(input)
#     print("Output shape:", output.shape)

#     d_out = np.random.randn(*output.shape)
#     d_input = model.backward(d_out)
#     print("Backward done. Mean of d_input:", np.mean(d_input))


# YOLOv4 neck: Spatial Pyramid Pooling (SPP)

## LocalMaxPooling

### Functions

In [21]:
def local_maxpool2d_forward_cpu(input4d, output4d, mask5d, kernel_size, stride, padding, dilation):
  for im in range(output4d.shape[0]):
    for c in range(output4d.shape[1]):
      for x in range(output4d.shape[2]):
        for y in range(output4d.shape[3]):
          max_val = -np.inf
          max_x, max_y = 0, 0
          for x_k in range(kernel_size):
            for y_k in range(kernel_size):
              x_i = x * stride + x_k - padding
              y_i = y * stride + y_k - padding
              if (0 <= x_i < output4d.shape[2]) and (0 <= y_i < output4d.shape[3]):
                val = input4d[im, c, x_i, y_i]
                if not np.isnan(max_val) and val > max_val:
                  max_val = val
                  max_x, max_y = x_i, y_i

          output4d[im, c, x, y] = max_val
          mask5d[im, c, x, y, 0] = max_x
          mask5d[im, c, x, y, 1] = max_y

def local_maxpool2d_backward_cpu(d_output4d, mask5d, d_input4d):
  for im in range(d_output4d.shape[0]):
    for c in range(d_output4d.shape[1]):
      for x in range(d_output4d.shape[2]):
        for y in range(d_output4d.shape[3]):
          mask = mask5d[im, c, x, y]
          max_x, max_y = mask[0], mask[1]
          d_input4d[im, c, max_x, max_y] += d_output4d[im, c, x, y]

### Classes

In [22]:
class LocalMaxPooling2d(NNLayer4D):
  def __init__(self, in_channels, kernel_size, stride, padding, dilation):
    super().__init__(in_channels)
    self.kernel_size = kernel_size
    self.stride = stride
    self.padding = padding
    self.dilation = dilation

    self.mask = None

  def forward(self, input):
    super().forward(input)

    output_shape = (input.shape[0], input.shape[1],
                    (input.shape[2] - (self.kernel_size - 1) * self.dilation + 2 * self.padding - 1) // self.stride + 1,
                    (input.shape[3] - (self.kernel_size - 1) * self.dilation + 2 * self.padding - 1) // self.stride + 1)
    output = np.zeros(output_shape)
    self.mask = np.zeros(output_shape + (2,), dtype=np.int32) # store location of pooling in 2D

    local_maxpool2d_forward_cpu(self.input, output, self.mask, self.kernel_size, self.stride, self.padding, self.dilation)
    self.output = output

    return output

  def backward(self, d_output):
    d_input = np.zeros_like(self.input)
    local_maxpool2d_backward_cpu(d_output, self.mask, d_input)

    #grad_clip(d_input)

    return d_input

## SPP

### Functions

In [23]:
def spp_concatenate_gpu(input1, input2, input3, input4, output): # axis = 1
  raise NotImplementedError

def spp_split_gpu(input, output1, output2, output3, output4): # axis = 1
  raise NotImplementedError

### Classes

In [24]:
class SPP(NNLayer4D):
  def __init__(self, in_channels):
    super().__init__(in_channels)
    self.maxpool5 = LocalMaxPooling2d(in_channels, 5, 1, 2, 1)
    self.maxpool9 = LocalMaxPooling2d(in_channels, 9, 1, 4, 1)
    self.maxpool13 = LocalMaxPooling2d(in_channels, 13, 1, 6, 1)
    self.conv = Conv2D(in_channels * 4, in_channels, 1, 1, 0, 1)
    self.output = None

  def forward(self, input):
    super().forward(input)
    assert not np.isnan(input).any(), "Array1 contains NaN!"
    output1 = self.maxpool5.forward(input)
    assert not np.isnan(output1).any(), "Array2 contains NaN!"
    output2 = self.maxpool9.forward(input)
    assert not np.isnan(output2).any(), "Array3 contains NaN!"
    output3 = self.maxpool13.forward(input)
    assert not np.isnan(output3).any(), "Array4 contains NaN!"

    output = None
    if CALC_FLAG == CPU_FLAG:
      output = np.concatenate([input, output1, output2, output3], axis=1)
    elif CALC_FLAG == GPU_FLAG:
      spp_concatenate_gpu(input, output1, output2, output3, self.output)

    assert not np.isnan(output).any(), "Array5 contains NaN!"

    output = self.conv.forward(output)
    assert not np.isnan(output).any(), "Array6 contains NaN!"

    self.output = output

    return output

  def backward(self, d_output):
    d_output = self.conv.backward(d_output)
    d_input, d_outmax5, d_outmax9, d_outmax13 = None, None, None, None
    if CALC_FLAG == CPU_FLAG:
      d_input, d_outmax5, d_outmax9, d_outmax13 = np.split(d_output, 4, axis=1)
    elif CALC_FLAG == GPU_FLAG:
      spp_split_gpu(d_output, d_input, d_outmax5, d_outmax9, d_outmax13)

    d_inmax5 = self.maxpool5.backward(d_outmax5)
    d_inmax9 = self.maxpool9.backward(d_outmax9)
    d_inmax13 = self.maxpool13.backward(d_outmax13)

    if CALC_FLAG == CPU_FLAG:
      d_input += d_inmax5 + d_inmax9 + d_inmax13
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # grad_clip(d_input)

    return d_input

# YOLOv4 neck: Path Aggregation Network (PAN)

## Sampling

### Functions

In [25]:
def interpolation2d_forward_cpu(input4d, output4d, scale_factor):
  # note: output4d[2H, 2W] vs input4d[H, W]
  for im in range(input4d.shape[0]):
    for c in range(input4d.shape[1]):
      for x in range(input4d.shape[2]):
        for y in range(input4d.shape[3]):
          input_val = input4d[im, c, x, y]
          for x_k in range(scale_factor):
            for y_k in range(scale_factor):
              x_i = x * scale_factor + x_k
              y_i = y * scale_factor + y_k
              # guaranteed to be inside output4d
              output4d[im, c, x_i, y_i] = input_val

def interpolation2d_backward_cpu(d_output4d, d_input4d, scale_factor):
  # note: similar to downsampling, except we add the gradient
  for im in range(d_input4d.shape[0]):
    for c in range(d_input4d.shape[1]):
      for x in range(d_input4d.shape[2]):
        for y in range(d_input4d.shape[3]):
          sum_grad = 0
          for x_k in range(scale_factor):
            for y_k in range(scale_factor):
              x_i = x * scale_factor + x_k
              y_i = y * scale_factor + y_k
              sum_grad += d_output4d[im, c, x_i, y_i]
          d_input4d[im, c, x, y] = sum_grad

def global_avgpool2d_forward_cpu(input4d, output4d, scale_factor):
  # note: output4d[H, W] vs input4d[2H, 2W]; input is guaranteed to be even.
  for im in range(output4d.shape[0]):
    for c in range(output4d.shape[1]):
      for x in range(output4d.shape[2]):
        for y in range(output4d.shape[3]):
          sum = 0
          for x_k in range(scale_factor):
            for y_k in range(scale_factor):
              x_i = x * scale_factor + x_k
              y_i = y * scale_factor + y_k
              sum += input4d[im, c, x_i, y_i]
          output4d[im, c, x, y] = sum / (scale_factor * scale_factor)

def global_avgpool2d_backward_cpu(d_output4d, d_input4d, scale_factor):
  # note: just like upsampling, but... yeah
  N = scale_factor ** 2
  for im in range(d_output4d.shape[0]):
    for c in range(d_output4d.shape[1]):
      for x in range(d_output4d.shape[2]):
        for y in range(d_output4d.shape[3]):
          for x_k in range(scale_factor):
            for y_k in range(scale_factor):
              x_i = x * scale_factor + x_k
              y_i = y * scale_factor + y_k
              d_input4d[im, c, x_i, y_i] = d_output4d[im, c, x, y] / N

### Classes

In [26]:
class Upsample(NNLayer4D):
  def __init__(self, in_channels, scale_factor):
    super().__init__(in_channels)
    self.scale_factor = scale_factor
    self.conv = Conv2D(in_channels, in_channels // scale_factor, 1, 1, 0, 1)

  def forward(self, input):
    # check
    super().forward(input)

    # execute: reduce then expand
    input1 = self.conv.forward(input)
    output = np.zeros((input1.shape[0], input1.shape[1], input1.shape[2] * self.scale_factor, input1.shape[3] * self.scale_factor))

    if CALC_FLAG == CPU_FLAG:
      interpolation2d_forward_cpu(input1, output, self.scale_factor)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    return output

  def backward(self, d_output):
    d_input = np.zeros((self.input.shape[0], in_channels // self.scale_factor, self.input.shape[2], self.input.shape[3]))
    if CALC_FLAG == CPU_FLAG:
      interpolation2d_backward_cpu(d_output, d_input, self.scale_factor)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    d_input = self.conv.backward(d_input)
    # grad_clip(d_input)

    return d_input

class Downsample(NNLayer4D):
  def __init__(self, in_channels, scale_factor):
    super().__init__(in_channels)
    self.scale_factor = scale_factor
    self.conv = Conv2D(in_channels, in_channels * scale_factor, 1, 1, 0, 1)

  def forward(self, input):
    # check
    super().forward(input)

    output = np.zeros((input.shape[0], input.shape[1], input.shape[2] // self.scale_factor, input.shape[3] // self.scale_factor))
    # execute: reduce then expand
    if CALC_FLAG == CPU_FLAG:
      global_avgpool2d_forward_cpu(input, output, self.scale_factor)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    output = self.conv.forward(output)

    return output

  def backward(self, d_output):
    d_output = self.conv.backward(d_output)
    d_input = np.zeros_like(self.input)

    if CALC_FLAG == CPU_FLAG:
      global_avgpool2d_backward_cpu(d_output, d_input, self.scale_factor)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # grad_clip(d_input)

    return d_input

## Fusion

### Functions

In [27]:
def fusion_concatenate_gpu(input1, input2, output):
  pass

### Classes

In [28]:
class Fusion(NNLayer4D):
  def __init__(self, in_channels):
    super().__init__(in_channels)
    self.conv = Conv2D(in_channels * 2, in_channels, 1, 1, 0, 1)

  def forward(self, input1, input2):
    # check
    super().forward(input1)
    super().forward(input2)
    if input1.shape != input2.shape:
      raise ValueError(f"Input shapes must be the same, got {input1.shape} and {input2.shape} instead.")

    output = None

    # execute: concat then 1x1 conv
    if CALC_FLAG == CPU_FLAG:
      output = np.concatenate([input1, input2], axis=1)
    elif CALC_FLAG == GPU_FLAG:
      output = np.zeros((input1.shape[0], self.in_channels*2, input1.shape[2], input1.shape[3]))
      fusion_concatenate_gpu(input1, input2, output)

    output = self.conv.forward(output)

    return output

  def backward(self, d_output):
    d_output = self.conv.backward(d_output)

    d_input1, d_input2 = None, None

    if CALC_FLAG == CPU_FLAG:
      d_input1, d_input2 = np.split(d_output, 2, axis=1)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # grad_clip(d_input1)
    # grad_clip(d_input2)

    return d_input1, d_input2

## PAN

### Functions

In [29]:
def add_gpu(input1, input2, output):
  pass

### Classes

In [30]:
class PAN:
  def __init__(self, in_c1, in_c2, in_c3):
    if in_c1*2 == in_c2 and in_c2*2 == in_c3:
      pass
    else:
      raise ValueError(f"Unexpected input channels: {in_c1}, {in_c2}, {in_c3}.")

    self.in_c1 = in_c1
    self.in_c2 = in_c2
    self.in_c3 = in_c3

    self.ups1 = Upsample(in_c3, 2)
    self.ups2 = Upsample(in_c2, 2)
    self.downs1 = Downsample(in_c1, 2)
    self.downs2 = Downsample(in_c2, 2)
    self.fusion_u1 = Fusion(in_c2)
    self.fusion_u2 = Fusion(in_c1)
    self.fusion_d1 = Fusion(in_c2)
    self.fusion_d2 = Fusion(in_c3)

  def forward(self, input1, input2, input3): #input1 is the smallest but with c3
    A1 = input1
    A2 = self.fusion_u1.forward(self.ups1.forward(A1), input2)
    A3 = self.fusion_u2.forward(self.ups2.forward(A2), input3)

    B3 = A3
    B2 = self.fusion_d1.forward(self.downs1.forward(B3), A2)
    B1 = self.fusion_d2.forward(self.downs2.forward(B2), A1)

    return B1, B2, B3

  def backward(self, d_B1o, d_B2o, d_B3o):
    d_B2i, d_A1o = self.fusion_d2.backward(d_B1o)
    d_B2i = self.downs2.backward(d_B2i)

    if CALC_FLAG == CPU_FLAG:
      d_B2o = d_B2i + d_B2o
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    d_B3i, d_A2o = self.fusion_d1.backward(d_B2o)
    d_B3i = self.downs1.backward(d_B3i)

    if CALC_FLAG == CPU_FLAG:
      d_B3o = d_B3i + d_B3o
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    d_A3 = d_B3o

    d_A2i, d_i3 = self.fusion_u2.backward(d_A3)
    d_A2i = self.ups2.backward(d_A2i)

    if CALC_FLAG == CPU_FLAG:
      d_A2o = d_A2i + d_A2o
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    d_A1i, d_i2 = self.fusion_u1.backward(d_A2o)
    d_A1i = self.ups1.backward(d_A1i)

    if CALC_FLAG == CPU_FLAG:
      d_A1o = d_A1i + d_A1o
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    d_i1 = d_A1o

    # grad_clip(d_i1)
    # grad_clip(d_i2)
    # grad_clip(d_i3)

    return d_i1, d_i2, d_i3

# YOLOv4 head: YOLOv3 detection heads

## Functions

In [31]:
def sigmoid_forward4d_cpu(input4d, output4d):
  for im in range(input4d.shape[0]):
    for c in range(input4d.shape[1]):
      for x in range(input4d.shape[2]):
        for y in range(input4d.shape[3]):
          output4d[im, c, x, y] = 1 / (1 + np.exp(-input4d[im, c, x, y]))

def sigmoid_backward4d_cpu(d_output4d, output4d, d_input4d):
  for im in range(d_input4d.shape[0]):
    for c in range(d_input4d.shape[1]):
      for x in range(d_input4d.shape[2]):
        for y in range(d_input4d.shape[3]):
          val = output4d[im, c, x, y]
          d_input4d[im, c, x, y] = d_output4d[im, c, x, y] * val * (1 - val)

def sigmoid_forward5d_cpu(input5d, output5d):
  for a in range(input5d.shape[0]):
    for b in range(input5d.shape[1]):
      for c in range(input5d.shape[2]):
        for d in range(input5d.shape[3]):
          for e in range(input5d.shape[4]):
            output5d[a, b, c, d, e] = 1 / (1 + np.exp(-input5d[a, b, c, d, e]))

def sigmoid_backward5d_cpu(d_output5d, output5d, d_input5d):
  for a in range(d_input5d.shape[0]):
    for b in range(d_input5d.shape[1]):
      for c in range(d_input5d.shape[2]):
        for d in range(d_input5d.shape[3]):
          for e in range(d_input5d.shape[4]):
            val = output5d[a, b, c, d, e]
            d_input5d[a, b, c, d, e] = d_output5d[a, b, c, d, e] * val * (1 - val)

def exp_forward_cpu(input4d, bounding_prior, output4d):
  for im in range(input4d.shape[0]):
    for c in range(input4d.shape[1]):
      for x in range(input4d.shape[2]):
        for y in range(input4d.shape[3]):
          output4d[im, c, x, y] = bounding_prior * np.exp(input4d[im, c, x, y])

def exp_backward_cpu(d_output4d, output4d, d_input4d):
  for im in range(d_input4d.shape[0]):
    for c in range(d_input4d.shape[1]):
      for x in range(d_input4d.shape[2]):
        for y in range(d_input4d.shape[3]):
          d_input4d[im, c, x, y] = d_output4d[im, c, x, y] * output4d[im, c, x, y]

def broadcast_add_cpu(input4d, input2d, output4d):
  for im in range(input4d.shape[0]):
    for c in range(input4d.shape[1]):
      for x in range(input4d.shape[2]):
        for y in range(input4d.shape[3]):
          output4d[im, c, x, y] = input4d[im, c, x, y] + input2d[x, y]

## Classes

In [32]:
class DetectionHead:
  def __init__(self, in_channels, num_classes, num_anchors, bboxH, bboxW):
    self.in_channels = in_channels
    self.num_classes = num_classes
    self.num_anchors = num_anchors
    self.out_channels = num_anchors * (5 + num_classes)

    self.bboxH = bboxH
    self.bboxW = bboxW

    self.input = None
    self.output = None

    self.nn = NN([
        Conv2D(in_channels, in_channels, 1, 1, 0, 1),
        Conv2D(in_channels, in_channels, 3, 1, 1, 1),
        Conv2D(in_channels, in_channels, 1, 1, 0, 1),
        Conv2D(in_channels, in_channels, 3, 1, 1, 1),
        Conv2D(in_channels, in_channels, 1, 1, 0, 1),
        Conv2D(in_channels, in_channels, 3, 1, 1, 1),
        Conv2D(in_channels, self.out_channels, 1, 1, 0, 1)
    ])

  def forward(self, input):
    self.input = input
    # conv pass
    output = self.nn.forward(input)

    # reshape & slice
    output = output.reshape((output.shape[0], self.num_anchors, 5 + self.num_classes, output.shape[2], output.shape[3]))
    output = output.transpose((0, 1, 3, 4, 2)) # N, A, C, H, W -> N, A, H, W, C
    output_x = output[:, :, :, :, 0]
    output_y = output[:, :, :, :, 1]
    output_w = output[:, :, :, :, 2]
    output_h = output[:, :, :, :, 3]
    output_conf = output[:, :, :, :, 4]
    output_class = output[:, :, :, :, 5:]

    # transform
    if CALC_FLAG == CPU_FLAG:
      sigmoid_forward4d_cpu(output_x, output_x)
      sigmoid_forward4d_cpu(output_y, output_y)
      exp_forward_cpu(output_w, self.bboxW, output_w)
      exp_forward_cpu(output_h, self.bboxH, output_h)
      sigmoid_forward4d_cpu(output_conf, output_conf)
      sigmoid_forward5d_cpu(output_class, output_class)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # offset
    grid_y = np.arange(output.shape[2]).reshape(output.shape[2], 1)
    grid_x = np.arange(output.shape[3]).reshape(1, output.shape[3])
    grid_y = np.tile(grid_y, (1, output.shape[3]))
    grid_x = np.tile(grid_x, (output.shape[2], 1))

    # broadcast add
    if CALC_FLAG == CPU_FLAG:
      broadcast_add_cpu(output_x, grid_x, output_x)
      broadcast_add_cpu(output_y, grid_y, output_y)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # re-concatenate
    output_x = output_x[..., np.newaxis]
    output_y = output_y[..., np.newaxis]
    output_w = output_w[..., np.newaxis]
    output_h = output_h[..., np.newaxis]
    output_conf = output_conf[..., np.newaxis]
    output = np.concatenate([output_x, output_y, output_w, output_h, output_conf, output_class], axis=4)
    output = output.reshape(output.shape[0], -1, output.shape[4]) # N, A * H * W, C

    self.output = output

    return output

  def backward(self, d_output):
    # re-concatenate
    assert d_output.shape[1] == self.num_anchors * self.input.shape[2] * self.input.shape[3], "Shape not matched!"
    output_shape_preres = (d_output.shape[0], self.num_anchors, self.input.shape[2], self.input.shape[3], d_output.shape[2])
    d_output = d_output.reshape(output_shape_preres)
    d_output_x = d_output[:, :, :, :, 0]
    d_output_y = d_output[:, :, :, :, 1]
    d_output_w = d_output[:, :, :, :, 2]
    d_output_h = d_output[:, :, :, :, 3]
    d_output_conf = d_output[:, :, :, :, 4]
    d_output_class = d_output[:, :, :, :, 5:]

    # broadcast add
    # d_out = d_in so nothing changes

    # transform
    prev_output = self.output.reshape(output_shape_preres)
    output_x = prev_output[:, :, :, :, 0]
    output_y = prev_output[:, :, :, :, 1]
    output_w = prev_output[:, :, :, :, 2]
    output_h = prev_output[:, :, :, :, 3]
    output_conf = prev_output[:, :, :, :, 4]
    output_class = prev_output[:, :, :, :, 5:]

    if CALC_FLAG == CPU_FLAG:
      exp_backward_cpu(d_output_w, output_w, d_output_w)
      exp_backward_cpu(d_output_h, output_h, d_output_h)
      sigmoid_backward4d_cpu(d_output_x, output_x, d_output_x)
      sigmoid_backward4d_cpu(d_output_y, output_y, d_output_y)
      sigmoid_backward4d_cpu(d_output_conf, output_conf, d_output_conf)
      sigmoid_backward5d_cpu(d_output_class, output_class, d_output_class)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # reshape & slice
    d_output_x = d_output_x[..., np.newaxis]
    d_output_y = d_output_y[..., np.newaxis]
    d_output_w = d_output_w[..., np.newaxis]
    d_output_h = d_output_h[..., np.newaxis]
    d_output_conf = d_output_conf[..., np.newaxis]
    d_output = np.concatenate([d_output_x, d_output_y, d_output_w, d_output_h, d_output_conf, d_output_class], axis=4)
    d_output = d_output.transpose((0, 1, 4, 2, 3)) # N, A, H, W, C -> N, A, C, H, W
    d_output = d_output.reshape((d_output.shape[0], self.num_anchors * (5 + self.num_classes), d_output.shape[3], d_output.shape[4]))

    # conv pass
    d_input = self.nn.backward(d_output)

    return d_input

# YOLOv4 loss

## Functions

### CIoU

In [33]:
import numpy as np
def ciou_loss_forward_cpu(pred3d, truth2d, anchor_loss2d, best_anchor_idx1d, eps=1e-5):
  for n in range(pred3d.shape[0]):
    best_iou_val = -np.nan
    best_iou_idx = 0
    for a in range(pred3d.shape[1]):
      # iou loss
      x_p, y_p, w_p, h_p = pred3d[n, a, :4]
      x_t, y_t, w_t, h_t = truth2d[n, :4]

      x1_p, y1_p, x2_p, y2_p = x_p - w_p / 2, y_p - h_p / 2, x_p + w_p / 2, y_p + h_p / 2
      x1_t, y1_t, x2_t, y2_t = x_t - w_t / 2, y_t - h_t / 2, x_t + w_t / 2, y_t + h_t / 2
      inter_area = max(0, min(x2_p, x2_t) - max(x1_p, x1_t)) * max(0, min(y2_p, y2_t) - max(y1_p, y1_t))
      union_area = w_p * h_p + w_t * h_t - inter_area
      iou = inter_area / (union_area + eps)
      if iou > best_iou_val:
        best_iou_val = iou
        best_iou_idx = a

      # distance penalty
      p2 = (x_p - x_t) ** 2 + (y_p - y_t) ** 2
      c2 = (min(x1_p, x1_t) - max(x2_p, x2_t)) ** 2 + (min(y1_p, y1_t) - max(y2_p, y2_t)) ** 2
      dist_penalty = p2 / c2

      # aspect ratio penalty
      v = 4 / (np.pi ** 2) * np.power(np.arctan(w_p / h_p) - np.arctan(w_t / h_t), 2)
      ar_penalty = v**2 / (1 - iou + v)

      # final loss
      loss = 1 - iou + dist_penalty + ar_penalty

      anchor_loss2d[n, a] = loss
    best_anchor_idx1d[n] = best_iou_idx

def ciou_loss_backward_cpu(d_anchor_loss2d, pred3d, truth2d, d_pred3d, eps=1e-5):
  # NOTE: even when d_truth is not needed, DO NOT prune out.
  for n in range(pred3d.shape[0]):
    for a in range(pred3d.shape[1]):
      # extract values
      x_p, y_p, w_p, h_p = pred3d[n, a, :4]
      x_t, y_t, w_t, h_t = truth2d[n, :4]

      # final loss
      d_out = d_anchor_loss2d[n, a]
      d_iou_out = - d_out
      d_dist = d_out
      d_ar = d_out

      # ar_iou
      v = 4 / (np.pi ** 2) * np.power(np.arctan(w_p / h_p) - np.arctan(w_t / h_t), 2)
      x1_p, y1_p, x2_p, y2_p = x_p - w_p / 2, y_p - h_p / 2, x_p + w_p / 2, y_p + h_p / 2
      x1_t, y1_t, x2_t, y2_t = x_t - w_t / 2, y_t - h_t / 2, x_t + w_t / 2, y_t + h_t / 2
      ia_x = min(x2_p, x2_t) - max(x1_p, x1_t)
      ia_y = min(y2_p, y2_t) - max(y1_p, y1_t)
      inter_area = max(0, ia_x) * max(0, ia_y)
      union_area = w_p * h_p + w_t * h_t - inter_area
      iou = inter_area / (union_area + eps)

      d_iou_ar = v**2 / (1 - iou + v)**2 * d_ar
      d_iou = d_iou_out + d_iou_ar

      # iou
      d_ua = - inter_area / (union_area + eps)**2 * d_iou

      d_w_p_ua = h_p
      d_h_p_ua = w_p
      d_w_t_ua = h_t
      d_h_t_ua = w_t

      d_ia_iou = 1 / (union_area + eps) * d_iou
      d_ia_ua = - d_ua
      d_ia = d_ia_iou + d_ia_ua
      d_ia_x, d_ia_y = 0, 0
      if ia_x > 0 and ia_y > 0:
        d_ia_x = ia_y * d_ia
        d_ia_y = ia_x * d_ia

      d_x2_p_ia_x = d_ia_x if x2_p < x2_t else 0
      d_x2_t_ia_x = d_ia_x if x2_p > x2_t else 0
      d_x1_p_ia_x = -d_ia_x if x1_p > x1_t else 0
      d_x1_t_ia_x = -d_ia_x if x1_p < x1_t else 0
      d_y2_p_ia_y = d_ia_y if y2_p < y2_t else 0
      d_y2_t_ia_y = d_ia_y if y2_p > y2_t else 0
      d_y1_p_ia_y = -d_ia_y if y1_p > y1_t else 0
      d_y1_t_ia_y = -d_ia_y if y1_p < y1_t else 0

      # distance
      p2 = (x_p - x_t) ** 2 + (y_p - y_t) ** 2
      c2_x1, c2_x2, c2_y1, c2_y2 = min(x1_p, x1_t), max(x2_p, x2_t), min(y1_p, y1_t), max(y2_p, y2_t)
      c2 = (c2_x1 - c2_x2) ** 2 + (c2_y1 - c2_y2) ** 2

      d_p2 = 1 / c2 * d_dist
      d_c2 = - p2 / c2**2 * d_dist

      d_x_p_p2 = 2 * (x_p - x_t) * d_p2
      d_x_t_p2 = - d_x_p_p2
      d_y_p_p2 = 2 * (y_p - y_t) * d_p2
      d_y_t_p2 = - d_y_p_p2

      d_c2_x1_c2 = 2 * (c2_x1 - c2_x2) * d_c2
      d_c2_x2_c2 = - d_c2_x1_c2
      d_c2_y1_c2 = 2 * (c2_y1 - c2_y2) * d_c2
      d_c2_y2_c2 = - d_c2_y1_c2

      d_x1_p_c2_x1 = d_c2_x1_c2 if x1_p < x2_t else 0
      d_x1_t_c2_x1 = d_c2_x1_c2 if x1_p > x2_t else 0
      d_x2_p_c2_x2 = d_c2_x2_c2 if x2_p > x2_t else 0
      d_x2_t_c2_x2 = d_c2_x2_c2 if x2_p < x2_t else 0
      d_y1_p_c2_y1 = d_c2_y1_c2 if y1_p < y2_t else 0
      d_y1_t_c2_y1 = d_c2_y1_c2 if y1_p > y2_t else 0
      d_y2_p_c2_y2 = d_c2_y2_c2 if y2_p > y2_t else 0
      d_y2_t_c2_y2 = d_c2_y2_c2 if y2_p < y2_t else 0

      # ar = v**2 / (1 - iou + v)
      d_v = (2 * v * (1 - iou + v) - (1 - iou) * v**2) / (1 - iou + v)**2 * d_ar
      # v = 4 / (np.pi ** 2) * np.power(vp - vt, 2)
      vp, vt = np.arctan(w_p / h_p), np.arctan(w_t / h_t)
      d_vp_v = 8 / (np.pi ** 2) * (vp - vt) * d_v
      d_vt_v = - d_vp_v

      d_w_p_vp = 1 / (1 + (w_p / h_p) ** 2) * (1 / h_p) * d_vp_v
      d_h_p_vp = 1 / (1 + (w_p / h_p) ** 2) * (-w_p / h_p ** 2) * d_vp_v
      d_w_t_vt = 1 / (1 + (w_t / h_t) ** 2) * (1 / h_t) * d_vt_v
      d_h_t_vt = 1 / (1 + (w_t / h_t) ** 2) * (-w_t / h_t ** 2) * d_vt_v

      # low-level, thanks gemini for helping with code alignment.
      d_x1_p = d_x1_p_ia_x + d_x1_p_c2_x1
      d_y1_p = d_y1_p_ia_y + d_y1_p_c2_y1
      d_x2_p = d_x2_p_ia_x + d_x2_p_c2_x2
      d_y2_p = d_y2_p_ia_y + d_y2_p_c2_y2
      d_x1_t = d_x1_t_ia_x + d_x1_t_c2_x1
      d_y1_t = d_y1_t_ia_y + d_y1_t_c2_y1
      d_x2_t = d_x2_t_ia_x + d_x2_t_c2_x2
      d_y2_t = d_y2_t_ia_y + d_y2_t_c2_y2

      d_x_p_x1_p = d_x1_p
      d_x_p_x2_p = d_x2_p
      d_x_t_x1_t = d_x1_t
      d_x_t_x2_t = d_x2_t
      d_y_p_y1_p = d_y1_p
      d_y_p_y2_p = d_y2_p
      d_y_t_y1_t = d_y1_t
      d_y_t_y2_t = d_y2_t
      d_w_p_x1_p = - d_x1_p / 2
      d_w_p_x2_p = d_x2_p / 2
      d_w_t_x1_t = - d_x1_t / 2
      d_w_t_x2_t = d_x2_t / 2
      d_h_p_y1_p = - d_y1_p / 2
      d_h_p_y2_p = d_y2_p / 2
      d_h_t_y1_t = - d_y1_t / 2
      d_h_t_y2_t = d_y2_t / 2

      d_x_p = d_x_p_x1_p + d_x_p_x2_p + d_x_p_p2
      d_x_t = d_x_t_x1_t + d_x_t_x2_t + d_x_t_p2
      d_y_p = d_y_p_y1_p + d_y_p_y2_p + d_y_p_p2
      d_y_t = d_y_t_y1_t + d_y_t_y2_t + d_y_t_p2
      d_w_p = d_w_p_x1_p + d_w_p_x2_p + d_w_p_vp + d_w_p_ua
      d_w_t = d_w_t_x1_t + d_w_t_x2_t + d_w_t_vt + d_w_t_ua
      d_h_p = d_h_p_y1_p + d_h_p_y2_p + d_h_p_vp + d_h_p_ua
      d_h_t = d_h_t_y1_t + d_h_t_y2_t + d_h_t_vt + d_h_t_ua

      # write back
      d_pred3d[n, a, :4] = (d_x_p, d_y_p, d_w_p, d_h_p)
      #d_truth3d[n, a, :4] = (d_x_t, d_y_t, d_w_t, d_h_t)

### Binary cross entropy

In [34]:
import numpy as np
def binary_cross_entropy_forward_cpu(pred3d, truth2d, output3d, eps=1e-5):
  for n in range(pred3d.shape[0]):
    for a in range(pred3d.shape[1]):
      for c in range(pred3d.shape[2]):
        pred = pred3d[n, a, c]
        truth = truth2d[n, c]

        # safe pred
        pred = min(max(pred, eps), 1 - eps)

        loss = - truth * np.log(pred) - (1 - truth) * np.log(1 - pred)
        output3d[n, a, c] = loss

def binary_cross_entropy_backward_cpu(d_output3d, pred3d, truth2d, d_pred3d, eps=1e-5):
  for n in range(pred3d.shape[0]):
    for a in range(pred3d.shape[1]):
      for c in range(pred3d.shape[2]):
        d_output = d_output3d[n, a, c]
        pred = pred3d[n, a, c]
        truth = truth2d[n, c]
        pred_safe = min(max(pred, eps), 1 - eps)

        d_pred = (- truth / pred_safe + (1 - truth) / (1 - pred_safe)) * d_output
        d_pred3d[n, a, c] = d_pred

### Total loss

In [35]:
import numpy as np
def total_loss_forward_cpu(unweighted_loss3d, best_anchor_idx1d, gamma_ciou, gamma_obj, gamma_noobj, gamma_cls, total_loss):
  total_loss[0] = 0
  for n in range(unweighted_loss3d.shape[0]):
    for a in range(unweighted_loss3d.shape[1]):
      # loss prep
      uloss_arr = unweighted_loss3d[n, a] # for GPU pull request from GMEM to cache
      ciou_loss, obj_loss, cls_loss = uloss_arr[0], uloss_arr[1], 0
      for c in range(2, unweighted_loss3d.shape[2]):
        cls_loss += uloss_arr[c]

      loss = 0
      # real calc
      if a == best_anchor_idx1d[n]:
        loss = gamma_ciou * ciou_loss + gamma_obj * obj_loss + gamma_cls * cls_loss
      else:
        loss = gamma_ciou * ciou_loss + gamma_noobj * obj_loss + gamma_cls * cls_loss
      total_loss[0] += loss

def total_loss_backward_cpu(loss, best_anchor_idx1d, gamma_ciou, gamma_obj, gamma_noobj, gamma_cls, d_unweighted_loss3d):
  for n in range(d_unweighted_loss3d.shape[0]):
    for a in range(d_unweighted_loss3d.shape[1]):
      d_ciou, d_obj, d_cls = 0, 0, 0
      if a == best_anchor_idx1d[n]:
        d_ciou = gamma_ciou * loss
        d_obj = gamma_obj * loss
        d_cls = gamma_cls * loss
      else:
        d_ciou = gamma_ciou * loss
        d_obj = gamma_noobj * loss
        d_cls = gamma_cls * loss

      for c in range(2, d_unweighted_loss3d.shape[2]):
        d_unweighted_loss3d[n, a, c] = d_cls
      d_unweighted_loss3d[n, a, 0] = d_ciou
      d_unweighted_loss3d[n, a, 1] = d_obj

## Classes

In [36]:
# supposed input shape [N, P, C] and label [N, C]
class YOLOv4Loss:
  def __init__(self, label2d):
    self.label = label2d

    self.loss = None
    self.input = None
    self.input_shape = None
    self.best_anchor_idx = None

    self.gamma_ciou = 5
    self.gamma_obj = 1
    self.gamma_noobj = 0.5
    self.gamma_cls = 1

  def forward(self, input): #inputs are 3d [N, A, C]
    # store
    self.input_shape = input.shape
    self.input = input

    # concatenate & split
    input_ciou = self.input[:, :, :4]
    input_bce = self.input[:, :, 4:]
    label_ciou = self.label[:, :4]
    label_bce = self.label[:, 4:]

    # ciou loss
    ciou_loss = np.zeros((self.input_shape[0], self.input_shape[1]))
    self.best_anchor_idx = np.zeros(self.input_shape[0])
    if CALC_FLAG == CPU_FLAG:
      ciou_loss_forward_cpu(input_ciou, label_ciou, ciou_loss, self.best_anchor_idx)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # binary cross-entropy loss
    bce_loss = np.zeros_like(input_bce)
    if CALC_FLAG == CPU_FLAG:
      binary_cross_entropy_forward_cpu(input_bce, label_bce, bce_loss)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # total loss
    unweighted_loss = np.concatenate([ciou_loss[:, :, np.newaxis], bce_loss], axis=2)
    self.loss = np.zeros((1,))
    if CALC_FLAG == CPU_FLAG:
      total_loss_forward_cpu(unweighted_loss, self.best_anchor_idx, \
                             self.gamma_ciou, self.gamma_obj, self.gamma_noobj, self.gamma_cls, self.loss)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    self.loss = self.loss[0]

    return self.loss

  def backward(self, d_loss):
    # total loss
    d_unweighted_loss = np.zeros((self.input_shape[0], self.input_shape[1], self.input_shape[2]-3))
    if CALC_FLAG == CPU_FLAG:
      total_loss_backward_cpu(d_loss, self.best_anchor_idx, self.gamma_ciou,\
                              self.gamma_obj, self.gamma_noobj, self.gamma_cls, d_unweighted_loss)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # binary cross-entropy loss
    d_bce = d_unweighted_loss[:, :, 1:]
    d_input_bce = np.zeros_like(d_bce)
    if CALC_FLAG == CPU_FLAG:
      binary_cross_entropy_backward_cpu(d_bce, self.input[:, :, 4:], self.label[:, 4:], d_input_bce)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # ciou loss
    d_ciou = d_unweighted_loss[:, :, 0]
    d_input_ciou = np.zeros((self.input.shape[0], self.input.shape[1], 4))
    if CALC_FLAG == CPU_FLAG:
      ciou_loss_backward_cpu(d_ciou, self.input[:, :, :4], self.label[:, :4], d_input_ciou)
    elif CALC_FLAG == GPU_FLAG:
      raise NotImplementedError

    # concatenate & split
    d_input = np.concatenate([d_input_ciou, d_input_bce], axis=2)

    return d_input

# YOLOv4, finalized

So we have finally finished constructing the basic components of YOLOv4, now we wrap it all up, see section Main.

Before we put a model into training, we have to do some other setups first:
- Synchronize dataset to model inputs and labels.
- (YOLOv4 only) Calculate original bounding box sizes in detection heads.

## Setup

### Functions

### Classes

## Main

In [37]:
class YOLOv4:
  def __init__(self, in_channels, num_classes, bboxH, bboxW, labels): # fully hard-coded
    # structure
    self.cspdn53 = CSPDarknet53(in_channels)
    self.spp = SPP(64) # deepest layer of CSPDarknet53 is always 1024.
    self.pan = PAN(16, 32, 64) # yeah...
    self.detects = [
        DetectionHead(16, num_classes, 3, bboxH, bboxW),
        DetectionHead(32, num_classes, 3, bboxH, bboxW),
        DetectionHead(64, num_classes, 3, bboxH, bboxW)
    ]
    self.loss = YOLOv4Loss(labels)
    self.num_anchors = None

  def forward(self, input):
    p3, p4, p5 = self.cspdn53.forward(input)
    assert not np.isnan(p3).any(), "Array contains NaN!"
    assert not np.isnan(p4).any(), "Array contains NaN!"
    assert not np.isnan(p5).any(), "Array contains NaN!"
    #print('CSP done.')

    p5 = self.spp.forward(p5)
    assert not np.isnan(p5).any(), "Array contains NaN!"
    #print('SPP done.')

    p5, p4, p3 = self.pan.forward(p5, p4, p3)
    assert not np.isnan(p5).any(), "Array contains NaN!"
    assert not np.isnan(p4).any(), "Array contains NaN!"
    assert not np.isnan(p3).any(), "Array contains NaN!"
    #print('PAN done.')

    p3, p4, p5 = self.detects[0].forward(p3), self.detects[1].forward(p4), self.detects[2].forward(p5)
    assert not np.isnan(p3).any(), "Array contains NaN!"
    assert not np.isnan(p4).any(), "Array contains NaN!"
    assert not np.isnan(p5).any(), "Array contains NaN!"
    #print('Detection done.')

    # at this point, p3 p4 p5 has shape [N, :, C]

    self.num_anchors = (p3.shape[1], p4.shape[1], p5.shape[1])
    pLoss = np.concatenate([p3, p4, p5], axis=1)
    assert not np.isnan(pLoss).any(), "Array contains NaN!"
    #print('Concatenation done.')

    loss = self.loss.forward(pLoss)
    assert not np.isnan(loss), "Loss = Nan!"
    #print('Loss done.')

    return loss

  def backward(self):
    d_pLoss = self.loss.backward(1)
    assert not np.isnan(d_pLoss).any(), "Loss = NaN!"
    #print('Loss backward done.')

    d_p3, d_p4, d_p5 = np.split(d_pLoss, [self.num_anchors[0], self.num_anchors[0] + self.num_anchors[1]], axis=1)
    d_p3, d_p4, d_p5 = self.detects[0].backward(d_p3), self.detects[1].backward(d_p4), self.detects[2].backward(d_p5)
    assert not np.isnan(d_p3).any(), "Array contains NaN!"
    assert not np.isnan(d_p4).any(), "Array contains NaN!"
    assert not np.isnan(d_p5).any(), "Array contains NaN!"
    #print('Detection backward done.')

    d_p5, d_p4, d_p3 = self.pan.backward(d_p5, d_p4, d_p3)
    assert not np.isnan(d_p5).any(), "Array contains NaN!"
    assert not np.isnan(d_p4).any(), "Array contains NaN!"
    assert not np.isnan(d_p3).any(), "Array contains NaN!"
    #print('PAN backward done.')

    d_p5 = self.spp.backward(d_p5)
    assert not np.isnan(d_p5).any(), "Array contains NaN!"
    #print('SPP backward done.')

    self.cspdn53.backward(d_p3, d_p4, d_p5)
    #print('CSP backward done.')

    # yeah there's no need for output anyways...

## Test

In [38]:
# Use CPU
CALC_FLAG = CPU_FLAG

# Parameters
batch_size = 1
in_channels = 3
input_h = input_w = 32
num_classes = 4
bboxH = bboxW = 1  # Anchor dimensions (relative scale)

# Dummy input [2, 3, 16, 16]
input_tensor = np.random.rand(batch_size, in_channels, input_h, input_w).astype(np.float64)

# Dummy labels: [x, y, w, h, obj_conf, class1, class2, class3, class4]
labels = np.random.rand(batch_size, 4 + 1 + num_classes).astype(np.float64)

# Instantiate YOLOv4
model = YOLOv4(
    in_channels=in_channels,
    num_classes=num_classes,
    bboxH=bboxH,
    bboxW=bboxW,
    labels=labels
)

# Forward pass
print("Running YOLOv4 forward pass...")
loss = model.forward(input_tensor)
print("Loss:", loss)

# Backward pass
print("Running YOLOv4 backward pass...")
model.backward()
print("Backward pass complete.")

del model

Running YOLOv4 forward pass...
Loss: 604.3666086247001
Running YOLOv4 backward pass...
Backward pass complete.


# ConvLSTM